In [1]:
import mlflow
import os

# Keeps everything internal to the container
mlflow.set_tracking_uri("file:///tmp/mlflow_data")

# Set as main project name
mlflow.set_experiment("Ames_Housing_Analysis") 

print(f"Tracking URI: {mlflow.get_tracking_uri()}")

/home/dzxu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/dzxu/.local/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/05 21:51:33 INFO mlflow.tracking.fluent: Experiment with name 'Ames_Housing_Analysis' does not exist. Creating a new experiment.


Tracking URI: file:///tmp/mlflow_data


In [2]:
import pandas as pd
import mlflow_utils as mltuti
from sklearn.model_selection import train_test_split

# Load the clean data
df = pd.read_csv('train_clean.csv')

# Define features and target
cols_to_drop = [col for col in df.columns if 'price' in col.lower() or col == 'Id']
X = df.drop(columns=cols_to_drop)
y = df['SalePrice']

# Perform 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12)

print(f"Features being used: {X.shape[1]}")
print(f"Training on {X_train.shape[0]} houses | Testing on {X_test.shape[0]} houses.")

Features being used: 301
Training on 1164 houses | Testing on 292 houses.


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import mlflow_utils as mltuti
import mlflow

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

with mltuti.start_mlflow_run("Ames_Housing_Analysis"):
    model = LinearRegression()
    model.fit(X_train, y_train_log)
    
    y_pred_log = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test_log, y_pred_log))
    r2 = r2_score(y_test_log, y_pred_log)
    
    # Logging
    mlflow.log_param("random_state", 12)
    mlflow.log_param("num_features", X_train.shape[1])
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    
    print(f"Linear Regression Results:")
    print(f"RMSE: {rmse:.4f}")
    print(f"R^2 Score: {r2:.4f}")

Linear Regression Results:
RMSE: 0.1209
R^2 Score: 0.9054


In [4]:
from sklearn.linear_model import Ridge

# Tuning parameter
alpha_val = 1.0

with mltuti.start_mlflow_run("Ames_Housing_Analysis"):
    ridge_model = Ridge(alpha=alpha_val)
    ridge_model.fit(X_train, y_train_log)
    
    y_pred_ridge = ridge_model.predict(X_test)
    rmse_ridge = np.sqrt(mean_squared_error(y_test_log, y_pred_ridge))
    r2_ridge = r2_score(y_test_log, y_pred_ridge)
    
    # Log new model details
    mlflow.log_param("model_type", "Ridge")
    mlflow.log_param("alpha", alpha_val)
    mlflow.log_metric("rmse", rmse_ridge)
    mlflow.log_metric("r2", r2_ridge)
    
    print(f"Ridge Results (Alpha={alpha_val}):")
    print(f"RMSE: {rmse_ridge:.4f}")
    print(f"R^2 Score: {r2_ridge:.4f}")

Ridge Results (Alpha=1.0):
RMSE: 0.1121
R^2 Score: 0.9186


In [5]:
from sklearn.linear_model import Ridge

# Define range of alphas to test
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]

# Start the loop
for a in alphas:
    with mltuti.start_mlflow_run(experiment_name="Ames_Housing_Analysis", 
                                 run_name=f"Ridge_Alpha_{a}"):
        
        # Initialize and train
        ridge_model = Ridge(alpha=a)
        ridge_model.fit(X_train, y_train_log)
        
        # Predict and score
        y_pred = ridge_model.predict(X_test)
        
        # Calculate metrics
        rmse = np.sqrt(mean_squared_error(y_test_log, y_pred))
        r2 = r2_score(y_test_log, y_pred)
        
        # Log parameters and metrics
        mlflow.log_param("alpha", a)
        mlflow.log_param("model_type", "Ridge")
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        
        print(f"Alpha: {a:6} | RMSE: {rmse:.4f} | R^2: {r2:.4f}")

Alpha:   0.01 | RMSE: 0.1206 | R^2: 0.9058
Alpha:    0.1 | RMSE: 0.1189 | R^2: 0.9085
Alpha:    1.0 | RMSE: 0.1121 | R^2: 0.9186
Alpha:   10.0 | RMSE: 0.1071 | R^2: 0.9257
Alpha:  100.0 | RMSE: 0.1090 | R^2: 0.9231
